# Stacking Ensemble with Pre-trained Models
Combine pre-trained models (RF, XGBoost, LightGBM) using Stacking Ensemble with meta-learner on combined dataset

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import joblib

# ML Libraries
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Setup Paths and Load Data

In [2]:
# Define paths
current_dir = Path.cwd().parent
data_path = current_dir / 'data' / 'processed' / 'combined_features_cleaned.csv'
models_dir = current_dir / 'models'
notebook_models_dir = current_dir / 'notebooks' / 'models'

print(f"Data path: {data_path}")
print(f"Models directory: {models_dir}")
print(f"Notebook models directory: {notebook_models_dir}")

# Load combined dataset
df = pd.read_csv(data_path)
print(f"\nDataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['viral'].value_counts())

Data path: c:\Users\jesly\OneDrive\viral-content-predictor\data\processed\combined_features_cleaned.csv
Models directory: c:\Users\jesly\OneDrive\viral-content-predictor\models
Notebook models directory: c:\Users\jesly\OneDrive\viral-content-predictor\notebooks\models

Dataset shape: (31749, 106)

Target distribution:
viral
0    23811
1     7938
Name: count, dtype: int64


## 3. Load Pre-trained Models

In [3]:
print("Loading pre-trained models...\n")

loaded_models = {}
model_sources = {}

# Try to load from notebooks/models first, then from models directory
model_files = {
    'rf': ['rf_tuned_gridsearch.joblib'],
    'xgb': ['xgb_tuned_randomized.joblib'],
    'lgbm': ['lgbm_randomized.joblib']
}

# Check both locations
search_dirs = [notebook_models_dir, models_dir]

for model_name, filenames in model_files.items():
    for filename in filenames:
        for search_dir in search_dirs:
            model_path = search_dir / filename
            if model_path.exists():
                try:
                    loaded_models[model_name] = joblib.load(model_path)
                    model_sources[model_name] = str(model_path)
                    print(f"✓ Loaded {model_name.upper()}: {model_path.name}")
                    break
                except Exception as e:
                    print(f"✗ Failed to load {filename}: {str(e)[:60]}")
        if model_name in loaded_models:
            break

print(f"\n{len(loaded_models)} pre-trained models loaded successfully")

# Check what happened
if 'rf' in loaded_models:
    print(f"  Random Forest - {loaded_models['rf'].__class__.__name__}")
if 'xgb' in loaded_models:
    print(f"  XGBoost - {loaded_models['xgb'].__class__.__name__}")
if 'lgbm' in loaded_models:
    print(f"  LightGBM - {loaded_models['lgbm'].__class__.__name__}")

Loading pre-trained models...

✓ Loaded RF: rf_tuned_gridsearch.joblib
✓ Loaded XGB: xgb_tuned_randomized.joblib
✓ Loaded LGBM: lgbm_randomized.joblib

3 pre-trained models loaded successfully
  Random Forest - RandomForestClassifier
  XGBoost - XGBClassifier
  LightGBM - LGBMClassifier


## 4. Prepare Data

In [4]:
# Prepare features and target
target_col = 'viral'
exclude_cols = ['track_id', 'track_name', 'artists', 'popularity', 'virality_score']

feature_cols = [col for col in df.columns if col not in exclude_cols and col != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
X = X.fillna(X.mean())

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nClass distribution:")
print(f"  Viral: {sum(y==1):,} ({100*sum(y==1)/len(y):.2f}%)")
print(f"  Non-Viral: {sum(y==0):,} ({100*sum(y==0)/len(y):.2f}%)")

Feature matrix shape: (31749, 100)
Target shape: (31749,)
Number of features: 100

Class distribution:
  Viral: 7,938 (25.00%)
  Non-Viral: 23,811 (75.00%)


## 5. Create Train-Test Split

In [5]:
# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining class distribution:")
print(f"  Viral: {sum(y_train==1):,}")
print(f"  Non-Viral: {sum(y_train==0):,}")

Training set: (25399, 100)
Test set: (6350, 100)

Training class distribution:
  Viral: 6,350
  Non-Viral: 19,049


## 5.5. Feature Alignment for Pre-trained Models

In [9]:
print("="*70)
print("STEP 1: FEATURE EXTRACTION FROM PRE-TRAINED MODELS")
print("="*70)

# Extract feature information from each model
model_features = {}

# Random Forest features
if 'rf' in loaded_models:
    try:
        rf_model = loaded_models['rf']
        # Try to get feature names from the model
        if hasattr(rf_model, 'feature_names_in_'):
            rf_features = list(rf_model.feature_names_in_)
        else:
            # If no feature names, use column indices from training
            rf_features = list(range(rf_model.n_features_in_))
        model_features['rf'] = rf_features
        print(f"✓ Random Forest: {len(rf_features)} features")
    except Exception as e:
        print(f"⚠ Could not extract RF features: {str(e)[:50]}")

# XGBoost features
if 'xgb' in loaded_models:
    try:
        xgb_model = loaded_models['xgb']
        if hasattr(xgb_model, 'feature_names_in_'):
            xgb_features = list(xgb_model.feature_names_in_)
        else:
            xgb_features = list(range(xgb_model.n_features_in_))
        model_features['xgb'] = xgb_features
        print(f"✓ XGBoost: {len(xgb_features)} features")
        if hasattr(xgb_model, 'feature_names_in_'):
            print(f"  Features: {xgb_features[:5]}...")
    except Exception as e:
        print(f"⚠ Could not extract XGB features: {str(e)[:50]}")

# LightGBM features
if 'lgbm' in loaded_models:
    try:
        lgbm_model = loaded_models['lgbm']
        if hasattr(lgbm_model, 'feature_names_in_'):
            lgbm_features = list(lgbm_model.feature_names_in_)
        else:
            lgbm_features = list(range(lgbm_model.n_features_in_))
        model_features['lgbm'] = lgbm_features
        print(f"✓ LightGBM: {len(lgbm_features)} features")
        if hasattr(lgbm_model, 'feature_names_in_'):
            print(f"  Features: {lgbm_features[:5]}...")
    except Exception as e:
        print(f"⚠ Could not extract LightGBM features: {str(e)[:50]}")

print(f"\n{'='*70}")
print("STEP 2: ALIGNING FEATURES FROM COMBINED DATASET")
print(f"{'='*70}\n")

# Create aligned feature datasets for each model
aligned_data = {}

def align_features_for_model(X_data, expected_features, model_name):
    """Align features to match what the model expects"""
    if isinstance(expected_features[0], int):
        # If features are indices, use positional indexing
        try:
            aligned = X_data.iloc[:, expected_features].copy()
            print(f"✓ {model_name}: Using positional features (indices 0-{max(expected_features)})")
            return aligned
        except Exception as e:
            print(f"⚠ {model_name}: Could not align by position: {str(e)[:50]}")
            return None
    else:
        # If features are names, try to match by name
        available = [f for f in expected_features if f in X_data.columns]
        missing = [f for f in expected_features if f not in X_data.columns]
        
        if missing:
            print(f"✓ {model_name}: Found {len(available)}/{len(expected_features)} features")
            if missing:
                print(f"  Missing {len(missing)}: {missing[:3]}...")
            # Use available features + pad with zeros for missing
            aligned = pd.DataFrame(index=X_data.index)
            for feat in expected_features:
                if feat in X_data.columns:
                    aligned[feat] = X_data[feat]
                else:
                    aligned[feat] = 0.0
            return aligned
        else:
            print(f"✓ {model_name}: All {len(available)} features found")
            return X_data[expected_features].copy()

# Align features for each model
for model_name, features in model_features.items():
    if model_name == 'rf':
        aligned = align_features_for_model(X_test, features, 'Random Forest')
        if aligned is not None:
            aligned_data['rf_test'] = aligned
            aligned = align_features_for_model(X_train, features, 'Random Forest (train)')
            if aligned is not None:
                aligned_data['rf_train'] = aligned
    elif model_name == 'xgb':
        aligned = align_features_for_model(X_test, features, 'XGBoost')
        if aligned is not None:
            aligned_data['xgb_test'] = aligned
            aligned = align_features_for_model(X_train, features, 'XGBoost (train)')
            if aligned is not None:
                aligned_data['xgb_train'] = aligned
    elif model_name == 'lgbm':
        aligned = align_features_for_model(X_test, features, 'LightGBM')
        if aligned is not None:
            aligned_data['lgbm_test'] = aligned
            aligned = align_features_for_model(X_train, features, 'LightGBM (train)')
            if aligned is not None:
                aligned_data['lgbm_train'] = aligned

print("\n✓ Feature alignment complete!")

STEP 1: FEATURE EXTRACTION FROM PRE-TRAINED MODELS
✓ Random Forest: 127 features
✓ XGBoost: 127 features
  Features: [np.str_('duration_ms'), np.str_('danceability'), np.str_('energy'), np.str_('key'), np.str_('loudness')]...
✓ LightGBM: 77 features
  Features: [np.str_('spectral_contrast_3_std'), np.str_('spectral_contrast_4_std'), np.str_('spectral_bandwidth_mean'), np.str_('spectral_contrast_2_std'), np.str_('spectral_rolloff_std')]...

STEP 2: ALIGNING FEATURES FROM COMBINED DATASET

✓ Random Forest: Found 9/127 features
  Missing 118: ['duration_ms', 'key', 'mode']...
✓ Random Forest (train): Found 9/127 features
  Missing 118: ['duration_ms', 'key', 'mode']...
✓ XGBoost: Found 9/127 features
  Missing 118: [np.str_('duration_ms'), np.str_('key'), np.str_('mode')]...
✓ XGBoost (train): Found 9/127 features
  Missing 118: [np.str_('duration_ms'), np.str_('key'), np.str_('mode')]...
✓ LightGBM: All 77 features found
✓ LightGBM (train): All 77 features found

✓ Feature alignment comp

## 6. Evaluate Individual Pre-trained Models

In [10]:
print("="*70)
print("EVALUATING INDIVIDUAL PRE-TRAINED MODELS")
print("="*70)

individual_results = {}

for model_name, model in loaded_models.items():
    try:
        print(f"\n{model_name.upper()} Model:")
        print("-" * 70)
        
        # Use aligned features for prediction
        test_data = aligned_data.get(f'{model_name}_test')
        if test_data is None:
            print(f"  ✗ No aligned features for {model_name}")
            continue
        
        # Get predictions using aligned features
        y_pred = model.predict(test_data)
        y_pred_proba = model.predict_proba(test_data)[:, 1]
        
        # Calculate metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc_score = roc_auc_score(y_test, y_pred_proba)
        
        individual_results[model_name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'auc': auc_score,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
        
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall:    {rec:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
        print(f"  AUC-ROC:   {auc_score:.4f}")
        
    except Exception as e:
        print(f"  ✗ Error evaluating {model_name}: {str(e)[:100]}")

EVALUATING INDIVIDUAL PRE-TRAINED MODELS

RF Model:
----------------------------------------------------------------------
  Accuracy:  0.7499
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  AUC-ROC:   0.5999

XGB Model:
----------------------------------------------------------------------
  Accuracy:  0.7499
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  AUC-ROC:   0.5741

LGBM Model:
----------------------------------------------------------------------
  Accuracy:  0.9509
  Precision: 0.9328
  Recall:    0.8659
  F1-Score:  0.8981
  AUC-ROC:   0.9592


## 7. Train Meta-Learner for Stacking Ensemble

In [ ]:
print("\n" + "="*70)
print("STEP 1: TRAINING META-LEARNER FOR STACKING")
print("="*70)

if len(individual_results) < 2:
    print("\n⚠ WARNING: Less than 2 models evaluated. Stacking requires at least 2 models.")
    meta_learner = None
else:
    print(f"\nGenerating meta-features from {len(individual_results)} base models...\n")
    
    # Step 1: Collect training predictions from base models on TRAINING data
    meta_features_train = []
    meta_model_names = []
    
    for model_name in individual_results.keys():
        # Get aligned training data for this model
        train_data = aligned_data.get(f'{model_name}_train')
        if train_data is not None:
            model = loaded_models[model_name]
            # Get probability predictions on training data
            train_proba = model.predict_proba(train_data)[:, 1]
            meta_features_train.append(train_proba)
            meta_model_names.append(model_name.upper())
            print(f"  ✓ Got training predictions from {model_name.upper()}")
    
    if meta_features_train:
        # Stack training predictions as meta-features
        X_train_meta = np.column_stack(meta_features_train)
        print(f"\n✓ Meta-features shape (train): {X_train_meta.shape}")
        print(f"  Base models: {', '.join(meta_model_names)}")
        
        # Step 2: Train meta-learner on meta-features
        print(f"\nTraining meta-learner (Logistic Regression)...")
        meta_learner = LogisticRegression(random_state=42, max_iter=1000, verbose=0)
        meta_learner.fit(X_train_meta, y_train)
        print("✓ Meta-learner trained!")
        
        # Show meta-learner weights
        print(f"\nMeta-learner model weights (feature importance):")
        for name, coef in zip(meta_model_names, meta_learner.coef_[0]):
            print(f"  {name:15}: {coef:7.4f}")
    else:
        print("✗ No valid predictions for meta-learner training")
        meta_learner = None


CREATING VOTING ENSEMBLE
✓ Added RandomForest to ensemble
✓ Added XGBoost to ensemble
✓ Added LightGBM to ensemble

Total models in ensemble: 3

✓ Voting Ensemble created successfully!
  Voting method: soft (probability-based)
  Number of base models: 3


## 8. Evaluate Stacking Ensemble

In [ ]:
print("\n" + "="*70)
print("STEP 2: EVALUATING STACKING ENSEMBLE")
print("="*70)

if meta_learner is not None and len(individual_results) > 0:
    try:
        print("\nGenerating meta-features from base model predictions on TEST data...\n")
        
        # Collect test predictions from base models
        meta_features_test = []
        
        for model_name in individual_results.keys():
            if f'{model_name}_test' in aligned_data:
                test_data = aligned_data[f'{model_name}_test']
                model = loaded_models[model_name]
                test_proba = model.predict_proba(test_data)[:, 1]
                meta_features_test.append(test_proba)
                print(f"  ✓ Got test predictions from {model_name.upper()}")
        
        if meta_features_test:
            # Stack test predictions as meta-features
            X_test_meta = np.column_stack(meta_features_test)
            print(f"\n✓ Meta-features shape (test): {X_test_meta.shape}")
            
            # Use meta-learner for final predictions
            print(f"\nUsing meta-learner for final predictions...")
            ensemble_pred_proba = meta_learner.predict_proba(X_test_meta)[:, 1]
            ensemble_pred = meta_learner.predict(X_test_meta)
            
            # Calculate metrics
            ensemble_acc = accuracy_score(y_test, ensemble_pred)
            ensemble_prec = precision_score(y_test, ensemble_pred)
            ensemble_rec = recall_score(y_test, ensemble_pred)
            ensemble_f1 = f1_score(y_test, ensemble_pred)
            ensemble_auc = roc_auc_score(y_test, ensemble_pred_proba)
            
            ensemble_results = {
                'accuracy': ensemble_acc,
                'precision': ensemble_prec,
                'recall': ensemble_rec,
                'f1': ensemble_f1,
                'auc': ensemble_auc,
                'y_pred': ensemble_pred,
                'y_pred_proba': ensemble_pred_proba
            }
            
            print("\nStacking Ensemble Performance (Test Set):")
            print("-" * 70)
            print(f"  Accuracy:  {ensemble_acc:.4f}")
            print(f"  Precision: {ensemble_prec:.4f}")
            print(f"  Recall:    {ensemble_rec:.4f}")
            print(f"  F1-Score:  {ensemble_f1:.4f}")
            print(f"  AUC-ROC:   {ensemble_auc:.4f}")
            
            print(f"\nClassification Report:")
            print(classification_report(y_test, ensemble_pred, target_names=['Non-Viral', 'Viral']))
        else:
            print("✗ No probability predictions available")
            ensemble_results = None
        
    except Exception as e:
        print(f"✗ Error evaluating stacking ensemble: {str(e)}")
        ensemble_results = None
else:
    print("✗ Meta-learner not trained or insufficient models for stacking")
    ensemble_results = None


EVALUATING VOTING ENSEMBLE

Manually aggregating predictions from base models with aligned features...
  ✓ Got predictions from RF
  ✓ Got predictions from XGB
  ✓ Got predictions from LGBM

Voting Ensemble Performance (Test Set):
----------------------------------------------------------------------
  Accuracy:  0.7556
  Precision: 0.9091
  Recall:    0.0252
  F1-Score:  0.0490
  AUC-ROC:   0.9458

Classification Report:
              precision    recall  f1-score   support

   Non-Viral       0.75      1.00      0.86      4762
       Viral       0.91      0.03      0.05      1588

    accuracy                           0.76      6350
   macro avg       0.83      0.51      0.45      6350
weighted avg       0.79      0.76      0.66      6350



## 9. Performance Comparison

In [ ]:
# Create comparison dataframe
comparison_data = []

for model_name, results in individual_results.items():
    comparison_data.append({
        'Model': model_name.upper(),
        'Accuracy': results['accuracy'],
        'Precision': results['precision'],
        'Recall': results['recall'],
        'F1-Score': results['f1'],
        'AUC-ROC': results['auc']
    })

if 'ensemble_results' in locals() and ensemble_results is not None:
    comparison_data.append({
        'Model': 'Stacking Ensemble',
        'Accuracy': ensemble_results['accuracy'],
        'Precision': ensemble_results['precision'],
        'Recall': ensemble_results['recall'],
        'F1-Score': ensemble_results['f1'],
        'AUC-ROC': ensemble_results['auc']
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*70)
print("MODEL PERFORMANCE COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

# Find best model
best_idx = comparison_df['AUC-ROC'].idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']
best_auc = comparison_df.loc[best_idx, 'AUC-ROC']

print(f"\n✓ Best Model (by AUC-ROC): {best_model_name}")
print(f"  AUC-ROC Score: {best_auc:.4f}")

## 10. Visualize Performance

In [ ]:
# Create comparison plots
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x_pos = np.arange(len(comparison_df))
width = 0.15

fig, ax = plt.subplots(figsize=(14, 6))

for i, metric in enumerate(metrics):
    ax.bar(x_pos + i*width, comparison_df[metric], width, label=metric)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison - Stacking Ensemble', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos + width * 2)
ax.set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(models_dir / 'stacking_ensemble_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comparison plot saved!")

## 11. ROC Curves

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
color_idx = 0

for model_name, results in individual_results.items():
    fpr, tpr, _ = roc_curve(y_test, results['y_pred_proba'])
    auc_score = results['auc']
    ax.plot(fpr, tpr, label=f"{model_name.upper()} (AUC = {auc_score:.4f})",
            linewidth=2, color=colors[color_idx])
    color_idx += 1

# Plot stacking ensemble ROC
if 'ensemble_results' in locals() and ensemble_results is not None:
    fpr, tpr, _ = roc_curve(y_test, ensemble_results['y_pred_proba'])
    auc_score = ensemble_results['auc']
    ax.plot(fpr, tpr, label=f"Stacking Ensemble (AUC = {auc_score:.4f})",
            linewidth=2.5, linestyle='--', color=colors[color_idx])

# Plot random classifier
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curves - Stacking Ensemble vs Pre-trained Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(models_dir / 'stacking_ensemble_roc.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ ROC curves plot saved!")

## 12. Confusion Matrices

In [ ]:
# Plot confusion matrices
n_models = len(individual_results) + (1 if 'ensemble_results' in locals() and ensemble_results is not None else 0)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

plot_idx = 0

for model_name, results in individual_results.items():
    if plot_idx < 4:
        ax = axes[plot_idx // 2, plot_idx % 2]
        cm = confusion_matrix(y_test, results['y_pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
        ax.set_title(f"{model_name.upper()}", fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
        plot_idx += 1

# Plot stacking ensemble confusion matrix
if 'ensemble_results' in locals() and ensemble_results is not None and plot_idx < 4:
    ax = axes[plot_idx // 2, plot_idx % 2]
    cm = confusion_matrix(y_test, ensemble_results['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax, cbar=False)
    ax.set_title('Stacking Ensemble', fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(models_dir / 'stacking_ensemble_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices plot saved!")

## 13. Save Stacking Ensemble

In [ ]:
if meta_learner is not None:
    try:
        ensemble_path = models_dir / 'stacking_ensemble_combined.joblib'
        joblib.dump(meta_learner, ensemble_path)
        print(f"✓ Stacking ensemble meta-learner saved to: {ensemble_path}")

        # Save ensemble info
        ensemble_info = {
            'type': 'StackingClassifier',
            'meta_learner': 'LogisticRegression',
            'n_base_models': len(individual_results),
            'base_models': list(individual_results.keys()),
            'test_accuracy': ensemble_results['accuracy'] if ensemble_results else None,
            'test_auc': ensemble_results['auc'] if ensemble_results else None,
            'feature_info': {
                'n_features': len(feature_cols),
                'features': feature_cols
            }
        }
        
        info_path = models_dir / 'stacking_ensemble_info.joblib'
        joblib.dump(ensemble_info, info_path)
        print(f"✓ Ensemble info saved to: {info_path}")
        
    except Exception as e:
        print(f"✗ Error saving ensemble: {str(e)}")
else:
    print("✗ Cannot save ensemble - meta-learner not trained")

## 14. Summary Report

In [ ]:
print("\n" + "="*70)
print("STACKING ENSEMBLE - FINAL SUMMARY")
print("="*70)

print(f"\nDataset Information:")
print(f"  Total samples: {len(df):,}")
print(f"  Total features: {len(feature_cols)}")
print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Class distribution (test set):")
print(f"    - Viral: {sum(y_test==1):,}")
print(f"    - Non-Viral: {sum(y_test==0):,}")

print(f"\nEnsemble Configuration:")
print(f"  Ensemble Type: Stacking Classifier")
print(f"  Meta-Learner: Logistic Regression")
print(f"  Base Models: {len(individual_results)}")
for name in individual_results.keys():
    print(f"    - {name.upper()}")

print(f"\nPerformance Summary (Test Set):")
print("-" * 70)
print(comparison_df.to_string(index=False))

if 'ensemble_results' in locals() and ensemble_results is not None:
    print(f"\n✓ Stacking Ensemble improvement over baseline models:")
    for model_name, results in individual_results.items():
        auc_diff = ensemble_results['auc'] - results['auc']
        acc_diff = ensemble_results['accuracy'] - results['accuracy']
        improvement_str = "↑" if auc_diff > 0 else "↓"
        print(f"  {model_name.upper():12} - AUC: {auc_diff:+.4f} {improvement_str}, Accuracy: {acc_diff:+.4f}")

print("\n" + "="*70)